In [1]:
# Load environment and suppress warnings
import os
import warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

if not api_key:
    raise ValueError("OPENAI_API_KEY not found.")

Key loaded: True


In [2]:
# Import required components and initialize deterministic model
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import (
    CommaSeparatedListOutputParser,
    JsonOutputParser,
    PydanticOutputParser,
)

model = ChatOpenAI(model="gpt-5-mini", temperature=0)

In [3]:
# Define a simple ticket input we will parse three different ways
ticket_text = """
Priority: High
Issue: Database timeout after deployment
Action: Restart service and monitor logs
Notes: Happens under peak load
"""

print(ticket_text)


Priority: High
Issue: Database timeout after deployment
Action: Restart service and monitor logs
Notes: Happens under peak load



In [4]:
# Apply CommaSeparatedListOutputParser to return a Python list
list_parser = CommaSeparatedListOutputParser()

list_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Extract structured information."),
        (
            "human",
            "Return exactly three values as a comma-separated list in this order:\n"
            "{format_instructions}\n\n"
            "Ticket:\n{ticket}",
        ),
    ]
)

list_chain = list_prompt | model | list_parser

tags = list_chain.invoke(
    {
        "ticket": ticket_text,
        "format_instructions": list_parser.get_format_instructions(),
    }
)

print("List result:", tags)
print("Type:", type(tags))

List result: ['High', 'Database timeout after deployment', 'Restart service and monitor logs']
Type: <class 'list'>


In [5]:
# Apply JsonOutputParser to return a Python dictionary
json_parser = JsonOutputParser()

json_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Extract structured information."),
        (
            "human",
            "Return JSON with keys: priority, issue, action.\n\n"
            "{format_instructions}\n\n"
            "Ticket:\n{ticket}",
        ),
    ]
)

json_chain = json_prompt | model | json_parser

ticket_json = json_chain.invoke(
    {
        "ticket": ticket_text,
        "format_instructions": json_parser.get_format_instructions(),
    }
)

print("JSON result:", ticket_json)
print("Type:", type(ticket_json))

JSON result: {'priority': 'High', 'issue': 'Database timeout after deployment', 'action': 'Restart service and monitor logs'}
Type: <class 'dict'>


In [6]:
# Apply PydanticOutputParser to enforce strict schema validation
from typing import Literal
from pydantic import BaseModel, Field

class Ticket(BaseModel):
    priority: Literal["Low", "Medium", "High"]
    issue: str
    action: str

pydantic_parser = PydanticOutputParser(pydantic_object=Ticket)

pydantic_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Extract structured information."),
        (
            "human",
            "Return a Ticket object matching this schema.\n\n"
            "{format_instructions}\n\n"
            "Ticket:\n{ticket}",
        ),
    ]
)

pydantic_chain = pydantic_prompt | model | pydantic_parser

ticket_obj = pydantic_chain.invoke(
    {
        "ticket": ticket_text,
        "format_instructions": pydantic_parser.get_format_instructions(),
    }
)

print("Pydantic result:", ticket_obj)
print("As dict:", ticket_obj.model_dump())
print("Type:", type(ticket_obj))

Pydantic result: priority='High' issue='Database timeout after deployment' action='Restart service and monitor logs'
As dict: {'priority': 'High', 'issue': 'Database timeout after deployment', 'action': 'Restart service and monitor logs'}
Type: <class '__main__.Ticket'>


In [7]:
# Compare the three outputs to highlight increasing strictness
print("ListOutputParser      ->", type(tags))
print("JsonOutputParser      ->", type(ticket_json))
print("PydanticOutputParser  ->", type(ticket_obj))

ListOutputParser      -> <class 'list'>
JsonOutputParser      -> <class 'dict'>
PydanticOutputParser  -> <class '__main__.Ticket'>
